In [1]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv('zeroshot.csv')

texts = [
    f"Title: {row['Title']}\nAbstract: {row['Content']}\nKeywords: {row['Keywords']}"
    for _, row in df.iterrows()
]

In [5]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

genres = [
    "Machine Learning / Deep Learning",
    "Computer Vision / 3D",
    "Natural Language Processing",
    "Theoretical Computer Science",
    "Computer Networks / Communication",
    "Software Engineering",
    "Databases",
    "Clustering",
    "Cloud / Server / API",
    "Privacy / Security",
    "Optimization / Algorithms",
    "Visualisation",
    "Data Structures",
    "Scheduling / Planning",
    "Mobile Applications",
    "Databases",
    "Robotics",
    "Mathematics / Theory",
    "Biology / Medicine",
    "Marketing / Business"
]


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0


In [6]:
batch_size = 32
labels = []
scores = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    results = classifier(batch, genres, multi_label=False)

    if isinstance(results, dict):
        results = [results]

    for r in results:
        labels.append(r['labels'])
        scores.append(r['scores'])

100%|██████████| 77/77 [1:15:39<00:00, 58.96s/it]


In [7]:
df['Labels'] = labels
df['Scores'] = scores
df.head()

,PID,Title,Keywords,Content,Labels,Scores
0,680998,2D Mapping Solutionsfor Low Cost Mobile Robot,[],"\nMapping, localization, and path-planning are...","[Robotics, Scheduling / Planning, Visualisatio...","[0.18993842601776123, 0.06088884174823761, 0.0..."
1,1903661,3D Scene Reconstruction using Diffusion Models...,"['computer vision', '3d reconstruction', 'text...",\nstate of the art text-to-image diffusion mod...,"[Computer Vision / 3D, Machine Learning / Deep...","[0.5174546241760254, 0.09498469531536102, 0.04..."
2,1229779,3D Shape Detection for Augmented Reality (3D f...,"['3d machine learning', 'computer vision', 'sh...","\nin previous work, 2d object recognition has ...","[Machine Learning / Deep Learning, Computer Vi...","[0.2877131998538971, 0.20900610089302063, 0.04..."
3,1888229,3D Shape Retrieval Meets Machine Learning : Im...,"['3d shape retrieval', 'pattern recognition', ...",\nin the rapidly evolving realm of cad model r...,"[Optimization / Algorithms, Machine Learning /...","[0.1169205754995346, 0.08399226516485214, 0.07..."
4,1245296,3D YOLO: End-to-End 3D Object Detection Using ...,"['computer vision', 'machine learning', 'auton...","\nfor safe and reliable driving, it is essenti...","[Machine Learning / Deep Learning, Computer Vi...","[0.36409062147140503, 0.221363365650177, 0.040..."


In [8]:
df.to_csv('zeroshot_inference.csv')

In [9]:
!cp zeroshot_inference.csv /content/drive/MyDrive/